# EchoProof R2-U2: final-step numerical diagnostic

This is **not training, resume, or checkpoint acceptance**. Use your existing prepared **Account B Tesla T4** runtime and your own Drive workspace. No packages are installed. Only this diagnostic folder and a new scratch directory receive new files. Original model weights, data, history, reports and source are read-only inputs.

Run these cells in order. Do not run the R2-U2 training notebook. Do not run another writer against this project concurrently. See README.md in this folder for scope and limitations.


In [ ]:
from pathlib import Path
import ast, json, subprocess, sys
from google.colab import drive
drive.mount('/content/drive')

# Change only if your actual Account B project folder has a different location.
PROJECT_ROOT = '/content/drive/MyDrive/Colab_R2_U1_Workspace'
# If the originals are inside your exported evidence folder, use:
# RUN_RELATIVE = 'R2_U2_Colab_Evidence/runs/r2/r2_u2_ast_b_baseline_v1'
RUN_RELATIVE = 'runs/r2/r2_u2_ast_b_baseline_v1'
ROOT = Path(PROJECT_ROOT).resolve()
DIAGNOSTIC = ROOT / 'diagnostics/r2_u2_final_step'
RUN = (ROOT / RUN_RELATIVE).resolve()
assert (ROOT / 'pyproject.toml').is_file(), f'Wrong project root: {ROOT}'
assert RUN.is_relative_to(ROOT) and RUN.is_dir(), f'Original run not found: {RUN}'
required = [DIAGNOSTIC / n for n in ['diagnose.py', 'test_safety.py', 'R2_U2_Final_Step_Diagnostic.ipynb']]
required += [RUN / n for n in ['latest.pt', 'best.pt', 'baseline.json', 'history.jsonl']]
missing = [str(p) for p in required if not p.is_file()]
assert not missing, 'Missing files: ' + repr(missing)
print('Project:', ROOT)
print('Original evidence:', RUN)
print('New reports only:', DIAGNOSTIC / 'results')


## Check the transferred diagnostic

This cell runs standard-library safety checks without loading audio, PyTorch or GPU models. A failure stops the notebook. These are not research acceptance tests.


In [ ]:
for name in ['diagnose.py', 'test_safety.py']:
    ast.parse((DIAGNOSTIC / name).read_text(encoding='utf-8'), filename=name)
safety = subprocess.run([sys.executable, '-B', str(DIAGNOSTIC / 'test_safety.py')], cwd=DIAGNOSTIC, text=True)
assert safety.returncode == 0, 'Safety checks failed; stop and send the error.'
print('Safety checks passed. The next cell performs the diagnostic, not training.')


## Execute the bounded diagnostic

Eight forward/backward conditions inspect the same final two training crops. No optimizer updates occur. The script checks the recorded environment and fails closed if it differs. It may download the pinned original model into Colab scratch. Hashing the existing audio shards can take several minutes on Drive; progress is printed. Do not interrupt merely because hashing is slow.

If a check fails, keep the diagnostic JSON and error. Do not install packages or edit the original recipe to bypass it.


In [ ]:
REPORTS = DIAGNOSTIC / 'results'
before = set(REPORTS.glob('diagnostic_*.json')) if REPORTS.exists() else set()
command = [sys.executable, '-B', str(DIAGNOSTIC / 'diagnose.py'), '--root', str(ROOT), '--run-dir', RUN_RELATIVE]
process = subprocess.run(command, cwd=ROOT)
after = set(REPORTS.glob('diagnostic_*.json')) if REPORTS.exists() else set()
created = sorted(after - before)
print('Exit code:', process.returncode)
print('Reports created:', [str(p) for p in created])
assert len(created) == 1, 'No unique completed report: send the console error; do not rerun training.'
REPORT_PATH = created[0]
report = json.loads(REPORT_PATH.read_text(encoding='utf-8'))
print('Status:', report['status'])
print('Original files unchanged:', report.get('original_files_unchanged'))
print('Conclusion:', report.get('conclusion', report.get('error')))
for case in report.get('cases', []):
    print(case['name'], 'nonfinite gradients:', case['scaled_nonfinite'],
          'scaler found inf:', case['scaler_found_inf'],
          'matches original:', case.get('matches_original', 'not applicable'))
print('This report does not accept the original checkpoint or change research policy.')


## Download and return the report

Run this cell even when the diagnostic reported a failed precondition. Send the downloaded JSON back for analysis. Keep the original run untouched. A diagnostic completion is not approval to retrain or publish.


In [ ]:
from google.colab import files
files.download(str(REPORT_PATH))
